In [17]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from typing import List, Tuple, Dict, Any, Optional
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI, AsyncOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from threading import Lock
import numpy as np
import logging
import os
import re
import time
import zipfile
import fitz
import ray
import base64
import asyncio
from opensearch_interface import get_opensearch_cluster_client, check_opensearch_index, insert_doc,insert_docs_bulk, delete_opensearch_index
from opensearch_interface import create_index_with_mapping
from dotenv import load_dotenv


In [18]:
load_dotenv()
logger = logging.getLogger(__name__)
OPENAI_MODEL = "gpt-4o-mini"
async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"
zip_file = os.path.join("data", "latest_dataset.zip")
EXTRACT_DIR = "data/unzipped"
BATCH_SIZE = 10
EMBEDDING_MODEL = "text-embedding-3-small"
HF_EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
MAX_IMAGE_SUMMARY_TOKENS = 300
MULTIMODAL_SUPPORT = False

# Initialize embeddings globally (will be recreated in Ray workers)
embeddings = HuggingFaceEmbeddings(model_name=HF_EMBEDDING_MODEL)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Extraction Layer

In [19]:
class ExtractionLayer:
    """
    Step 1: Extraction Layer
    Responsible for extracting text and images from PDF documents.
    Handles both text content and visual content extraction with summarization.
    """
    
    # @staticmethod
    # def summarize_images(pdf_path: str, images: List[Tuple]) -> Dict[int, str]:
    #     """
    #     Summarize multiple images in parallel using concurrent API calls.
    #     Much faster than sequential summarization.
        
    #     Args:
    #         pdf_path: Path to the PDF document
    #         images: List of image tuples from page.get_images()
            
    #     Returns:
    #         Dictionary mapping image index to summary
    #     """
    #     if not images:
    #         return {}
        
    #     image_summaries = {}
        
    #     try:
    #         pdf_doc = fitz.open(pdf_path)
            
    #         # Process all images in parallel using async task
    #         async def process_image(idx, image_info):
    #             try:
    #                 xref = image_info[0]
    #                 pix = fitz.Pixmap(pdf_doc, xref)
                    
    #                 # Convert CMYK to RGB if needed
    #                 if pix.n - pix.alpha > 3:
    #                     pix = fitz.Pixmap(fitz.csRGB, pix)
                    
    #                 # Convert to PNG bytes for API
    #                 img_data = pix.tobytes("png")
    #                 base64_image = base64.b64encode(img_data).decode("utf-8")
                    
    #                 response = await async_client.chat.completions.create(
    #                     model=OPENAI_MODEL,
    #                     messages=[
    #                         {
    #                             "role": "user",
    #                             "content": [
    #                                 {
    #                                     "type": "text",
    #                                     "text": "Describe what you see in this image and summarize its content briefly."
    #                                 },
    #                                 {
    #                                     "type": "image_url",
    #                                     "image_url": {
    #                                         "url": f"data:image/png;base64,{base64_image}"
    #                                     }
    #                                 }
    #                             ]
    #                         }
    #                     ],
    #                     max_tokens=MAX_IMAGE_SUMMARY_TOKENS
    #                 )
    #                 return response.choices[0].message.content
    #             except Exception as e:
    #                 logger.warning(f"[EXTRACTION] Failed to summarize image {idx}: {e}")
    #                 return idx, "Image summary unavailable"
            
    #         tasks = [process_image(idx, info) for idx, info in enumerate(images)]
            
    #         image_summaries = tasks
            
    #         pdf_doc.close()
            
    #     except Exception as e:
    #         logger.error(f"Error summarizing images batch: {e}")
        
    #     return image_summaries
    
    @staticmethod
    def extract_from_zipfile(zip_file: str) -> List[str]:
        """
        Extract files from a zip archive with optimization.
        
        - Only extracts PDF files (skips other file types)
        - Checks if files already exist to avoid re-extraction
        - Uses parallel extraction for faster performance
        
        Args:
            zip_file: Path to the zip file
            
        Returns:
            List of PDF file names that were extracted
        """
        try:
            # Create extract directory if it doesn't exist
            Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)
            
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                # Get all file names
                all_files = zip_ref.namelist()
                
                # Filter only PDF files and check which ones need extraction
                pdf_files = [f for f in all_files if f.lower().endswith('.pdf')]
                files_to_extract = []
                
                for pdf_file in pdf_files:
                    extracted_path = Path(EXTRACT_DIR) / pdf_file
                    # Only extract if file doesn't already exist or is outdated
                    if not extracted_path.exists():
                        files_to_extract.append(pdf_file)
                
                # If no files need extraction, return existing PDFs
                if not files_to_extract:
                    logger.info(f"[EXTRACTION] All {len(pdf_files)} PDF files already exist in {EXTRACT_DIR}")
                    return pdf_files
                
                # Extract only PDF files in parallel
                def extract_file(file_name: str) -> str:
                    """Extract a single file from the zip."""
                    try:
                        zip_ref.extract(file_name, EXTRACT_DIR)
                        return file_name
                    except Exception as e:
                        logger.warning(f"[EXTRACTION] Failed to extract {file_name}: {e}")
                        return None
                
                # Use ThreadPoolExecutor for parallel extraction (max 4 threads)
                with ThreadPoolExecutor(max_workers=4) as executor:
                    extracted_results = list(executor.map(extract_file, files_to_extract))
                    extracted_files = [f for f in extracted_results if f is not None]
                
                logger.info(
                    f"[EXTRACTION] Extracted {len(extracted_files)} new PDF files from {zip_file} "
                    f"(Total PDFs: {len(pdf_files)}, Already existed: {len(pdf_files) - len(files_to_extract)})"
                )
                return pdf_files  # Return all PDF files, not just newly extracted ones
                
        except Exception as e:
            logger.error(f"Error extracting zip file: {e}")
            return []
    
    @staticmethod
    def extract_text_and_images(files: List[str]) -> Dict[str, Dict[str, Any]]:
        """
        Extract text and images from PDF files.
        
        Args:
            files: List of file names to process
            
        Returns:
            Dictionary with extracted content including text, images, and page metadata
        """
        doc_dictionary = {}
        
        for file in files:
            if not file.endswith(".pdf"):
                continue
            
            try:
                logger.info(f"[EXTRACTION] Processing file: {file}")
                pdf_path = EXTRACT_DIR + "/" + file
                pdf_doc = fitz.open(pdf_path)
                
                try:
                    file_data = {
                        "file_name": file,
                        "pages": []
                    }
                    
                    for page_num, page in enumerate(pdf_doc, start=1):
                        text = page.get_text("text")
                        tabs = page.find_tables()
                        images = page.get_images()
                        
                        # Process text blocks
                        if tabs.tables:
                            for tab in tabs.tables:
                                # Extract table content as a list of lists (rows/cols)
                                table_data = tab.extract()
                                # Convert table to string format and append to text
                                table_text = "\n".join([" | ".join([str(cell) if cell else "" for cell in row]) for row in table_data])
                                text += "\n\n[TABLE]\n" + table_text + "\n[/TABLE]\n"                       

                        page_data = {
                            "page_number": page_num,
                            "text": text,
                            "images": [],
                        }
                        
                        # Process images in batch (parallel)
                        # if images and MULTIMODAL_SUPPORT == True:
                        #     image_summaries = asynio.run(ExtractionLayer.summarize_images(str(Path(pdf_path)), images))
                        #     for idx, image_info in enumerate(images):
                        #         image_name = f"{file}_page_{page_num}_image_{idx+1}"
                        #         image_summary = image_summaries.get(idx, "Image summary unavailable")
                        #         page_data["images"].append({
                        #             "image_name": image_name,
                        #         })
                        #         page_data["text"] += f"\nImage {image_name}: {image_summary}"
                        
                        file_data["pages"].append(page_data)
                    
                    doc_dictionary[file] = file_data
                    logger.info(f"[EXTRACTION] Completed extraction for {file}")
                finally:
                    pdf_doc.close()
            except Exception as e:
                logger.error(f"[EXTRACTION] Error processing file {file}: {e}")
        
        return doc_dictionary

## Transform

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Constants
TEXT_CHUNK_SIZE = 1000
TEXT_CHUNK_OVERLAP = 100

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=TEXT_CHUNK_SIZE,
    chunk_overlap=TEXT_CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", "?", "!", ","]
)


class ChunkingPreprocessingLayer:
    """
    Step 2: Chunking & Preprocessing
    Responsible for splitting text into manageable chunks and cleaning content.
    Applies multiple chunking strategies for optimal results.
    """
    
    @staticmethod
    def clean_text(text: str) -> str:
        """
        Clean and normalize text by removing unwanted line breaks and spaces.
        
        Args:
            text: Raw text to clean
            
        Returns:
            Cleaned text
        """
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)  # Fix hyphenated line breaks
        text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)  # Single newlines to spaces
        text = re.sub(r"\n{2,}", "\n", text)  # Multiple newlines to single
        return text.strip()
    
    @staticmethod
    def get_chunks_from_text(text: str) -> List[str]:
        """
        Split text into chunks using the configured text splitter.
        
        Args:
            text: Text to split
            
        Returns:
            List of text chunks
        """
        cleaned_text = ChunkingPreprocessingLayer.clean_text(text)
        if not cleaned_text:
            return []
        return text_splitter.split_text(cleaned_text)
    
    @staticmethod
    def preprocess_chunks(chunks: List[str], file_name: str, page_num: int, version: str, image_name: str) -> List[Dict[str, Any]]:
        """
        Preprocess chunks and add structural metadata.
        
        Args:
            chunks: List of text chunks
            file_name: Name of source file
            page_num: Page number
            
        Returns:
            List of preprocessed chunk dictionaries with metadata
        """
        metadata = []
        for chunk_idx, chunk in enumerate(chunks):
            metadata.append({
                "version": version,
                "document_name": file_name,
                "page_number": page_num,
                "chunk_number": chunk_idx
            })
        return metadata



class EmbeddingManager:
    """
    Step 3: Embedding Generation
    Responsible for creating semantic embeddings for text chunks.
    Uses sentence transformers or OpenAI embeddings model.
    """
    
    @staticmethod
    def embed_text(text: str) -> Optional[List[float]]:
        """
        Generate embedding for a single text chunk.
        
        Args:
            text: Text chunk to embed
            
        Returns:
            Embedding vector or None if error
        """
        try:
            response = embeddings.embed_documents([text])
            return response[0] if response else None
        except Exception as e:
            logger.error(f"[EMBEDDING] Error generating embedding: {e}")
            return None
           

## Load

In [21]:
class IndexManager:
    def __init__(self, index_name, host, port, user_name, password):
        self.index_name = index_name
        self.opensearch_client =  get_opensearch_cluster_client(index_name, host, port, user_name, password)
   

        # delete_opensearch_index(opensearch_client, index_name)
        exists = check_opensearch_index(self.opensearch_client, self.index_name)
    
        if exists:
            print("[OPENSEARCH]Exist deleting OpenSearch index")
            delete_opensearch_index(self.opensearch_client, self.index_name)
            
        print("[OPENSEARCH]Creating OpenSearch index")
        success = create_index_with_mapping(self.opensearch_client, index_name)
        if success:
            print("[OPENSEARCH]Created OpenSearch index mapping")
        else:
            print("[OPENSEARCH]Failed")

    def insert(self, embeddings, metadatas, docs):

        success_count, failure_count = insert_doc(self.opensearch_client, self.index_name, docs, metadatas, embeddings)

        print(f"[OPENSEARCH]Inserted documents success count {success_count} and failure count {failure_count}")

    def insert_bulk(self, bulk_docs):
        success_count, failure_count = insert_docs_bulk(self.opensearch_client, bulk_docs)

        print(f"[OPENSEARCH]Inserted bulk documents success count {success_count} and failure count {failure_count}")

In [ ]:
index_name = "scalable-rag-index3"
host = 'localhost'
port = 9200
user_name = "admin"
password = "xxxxxxx"
index = IndexManager(index_name, host, port, user_name, password)



[OPENSEARCH]Exist deleting OpenSearch index
2026-05-14 19:48:58.684 | INFO     | opensearch_interface:delete_opensearch_index:143 - Trying to delete index scalable-rag-index3
2026-05-14 19:48:58.789 | INFO     | opensearch_interface:delete_opensearch_index:146 - Index scalable-rag-index3 deleted
[OPENSEARCH]Creating OpenSearch index
[OPENSEARCH]Created OpenSearch index mapping


## Ray Cluster

In [23]:
# Initialize Ray
ray.init(ignore_reinit_error=True)

2026-05-14 19:48:59,215	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


Python version:,3.12.7
Ray version:,2.55.1


In [24]:
@ray.remote
def extract_text_and_images_remote(files: List[str]) -> Dict[str, Dict[str, Any]]:
    """Ray remote wrapper for text and image extraction."""
    # Create a new instance inside the Ray worker to avoid serialization issues
    return ExtractionLayer.extract_text_and_images(files)


In [25]:
@ray.remote
def get_all_chunks(extracted_content: Dict[str, Dict[str, Any]]) -> Tuple[List[str], List[Dict[str, Any]]]:
    """Ray remote wrapper for chunking and preprocessing."""
    all_chunks = []
    all_metadatas = []
    for doc_name, doc_data in extracted_content.items():
        for page in doc_data.get("pages", []):
            chunks = ChunkingPreprocessingLayer.get_chunks_from_text(page["text"])
            metadatas = ChunkingPreprocessingLayer.preprocess_chunks(
                chunks, doc_name, page["page_number"], "v1.0", ""
            )
            all_chunks.extend(chunks)
            all_metadatas.extend(metadatas)
    return all_chunks, all_metadatas


In [26]:

@ray.remote
class EmbeddingWorker:
    def __init__(self):
        self.embedding_model = HuggingFaceEmbeddings(model_name=HF_EMBEDDING_MODEL)

    def embed_documents(self, documents):
        embed = []
        for document in documents:
            response = self.embedding_model.embed_documents([document])
            embed.append(response[0])
        return embed
    

In [27]:
@ray.remote
class IndexWorker:
    def __init__(self, index_name):
        self.index_name = index_name
        self.opensearch_client =  get_opensearch_cluster_client(index_name, host, port, user_name, password)
   
    def get_bulk_data(self, embeddings, metadatas, docs):
        bulk_data = []
        for doc, meta, vector in zip(docs, metadatas, embeddings):
            # Add a document to the index.
            document = {
                '_op_type': 'index',
                '_index': index_name,
                '_id': str(meta['document_name']) + "_" + str(meta['page_number']) + "_" + str(meta['chunk_number']),
                '_source': {
                    'vector_field': vector,
                    'metadata': meta,
                    'text': doc
                }
            }

            bulk_data.append(document)

        return bulk_data

## Pipeline

In [28]:
def batcher(iterable, batch_size: int):
    """
    Batch an iterable into chunks of specified size.
    
    Args:
        iterable: Iterable to batch
        batch_size: Size of each batch
        
    Yields:
        Lists of items in batches
    """
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [29]:
def embed_all_chunks(all_chunks):
    num_workers = 4

    # Start the workers
    embedding_workers = [EmbeddingWorker.remote() for _ in range(num_workers)]
    # Create a pool of worker threads (e.g., 5 threads)

    batch_size = len(all_chunks)//num_workers
    # Submit each batch as a separate task to the thread pool
    document_chunks = [all_chunks[i:i+batch_size] for i in range(0, len(all_chunks), batch_size)]

    # Perform embedding generation in parallel
    start_time = time.time()
    embedding_tasks = [worker.embed_documents.remote(chunk) for worker, chunk in zip(embedding_workers, document_chunks)]
    embeddings = ray.get(embedding_tasks)
    end_time = time.time()

    # Flatten the embeddings list
    embeddings = [embedding for sublist in embeddings for embedding in sublist]

    print(f"[EMBEDDING] length of embedding: {len(embeddings)}")
    print(f"[EMBEDDING] Total execution time: {end_time - start_time} seconds")
    return embeddings

In [31]:
def insert_into_opensearch(batch_embeddings, batch_metadatas, batch_chunks):

    # Start the workers
    # insert_workers = IndexWorker.remote(index_name)
    
    # Perform inserting generation in parallel
    start_time = time.time()
    # bulk_data = insert_workers.get_bulk_data.remote(batch_embeddings, batch_metadatas, batch_chunks)
    # bulk_data = ray.get(bulk_data)
    # print(f"Bulk data {len(bulk_data)}")
    # index.insert_bulk(bulk_data)
    index.insert(batch_embeddings, batch_metadatas, batch_chunks)
    end_time = time.time()

    print(f"[OPENSEARCH] Total execution time: {end_time - start_time} seconds")
    

In [32]:
def run_pipeline():
    start_time = time.time()

    files = ExtractionLayer.extract_from_zipfile(zip_file)
    print(f"[PIPELINE] {len(files)} files to be extracted")
    
    batch_generate = batcher(files, batch_size=BATCH_SIZE)

    for batch_number, batch_files in enumerate(batch_generate, start=1):
        batch_chunks, batch_metadatas, batch_embeddings = [], [], []
        # Extract text and images using Ray
        documents = extract_text_and_images_remote.remote(batch_files)
        batch_docs = ray.get(documents)
        
        # Get chunks using Ray
        chunks_ref = get_all_chunks.remote(batch_docs)
        batch_chunks, batch_metadatas = ray.get(chunks_ref)
        print(f"[PIPELINE] Total chunks and metadata created: {len(batch_chunks)}, {len(batch_metadatas)}")
    
        # Embed chunks in batches using Ray
        print(f"[EMBEDDING] Starting parallel embedding for {len(batch_chunks)} chunks...")
        batch_embeddings = embed_all_chunks(batch_chunks)
        print(f"[EMBEDDING] completed parallel embedding for batch {batch_number} with total {len(batch_embeddings)} embedding.")
    
        # Insert into OpenSearch        
        print(f"[OPENSEARCH] Inserting {len(batch_chunks)} documents...")
        insert_into_opensearch(batch_embeddings, batch_metadatas, batch_chunks)            
        
    end_time = time.time()
    print(f"[PIPELINE] Total execution time: {end_time - start_time} seconds")


In [33]:
run_pipeline()

[PIPELINE] 1076 files to be extracted
[PIPELINE] Total chunks and metadata created: 452, 452
[EMBEDDING] Starting parallel embedding for 452 chunks...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6296.33it/s]


[EMBEDDING] length of embedding: 452
[EMBEDDING] Total execution time: 100.53016448020935 seconds
[EMBEDDING] completed parallel embedding for batch 1 with total 452 embedding.
[OPENSEARCH] Inserting 452 documents...
2026-05-14 19:51:41.845 | INFO     | opensearch_interface:insert_doc:176 - {'_index': 'scalable-rag-index3', '_id': 'aa9edb6b15f25884ea1d47308c271fbf5badd29d.pdf_1_0', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1}
2026-05-14 19:51:41.908 | INFO     | opensearch_interface:insert_doc:176 - {'_index': 'scalable-rag-index3', '_id': 'aa9edb6b15f25884ea1d47308c271fbf5badd29d.pdf_1_1', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1}
2026-05-14 19:51:41.967 | INFO     | opensearch_interface:insert_doc:176 - {'_index': 'scalable-rag-index3', '_id': 'aa932e9b0226461291e5f75cc35d2e13

In [34]:
# Shutdown Ray
ray.shutdown() 